# 🤚 Rock Paper Scissors — CNN con Data Augmentation
Classificazione di gesti della mano con una rete neurale convoluzionale.

In [ ]:

!pip install tensorflow matplotlib numpy pillow

In [ ]:

import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.callbacks import Callback


## 1. Download e Estrazione Dataset

In [ ]:
# ── Download dataset da URL ufficiale TensorFlow ──────────────────────────────
import urllib.request

urls = {
    'rps.zip':          'https://storage.googleapis.com/download.tensorflow.org/data/rps.zip',
    'rps-test-set.zip': 'https://storage.googleapis.com/download.tensorflow.org/data/rps-test-set.zip'
}

for filename, url in urls.items():
    if not os.path.exists(filename):
        print(f'Scaricando {filename}...')
        urllib.request.urlretrieve(url, filename)
    with zipfile.ZipFile(filename, 'r') as z:
        z.extractall('.')
    print(f'{filename} estratto ✓')

TRAIN_DIR = 'rps'
TEST_DIR  = 'rps-test-set'

# Verifica struttura cartelle
for split, path in [('Train', TRAIN_DIR), ('Test', TEST_DIR)]:
    for cls in os.listdir(path):
        n = len(os.listdir(os.path.join(path, cls)))
        print(f'{split}/{cls}: {n} immagini')

## 2. Data Pipeline e Augmentation (Fase A)

In [ ]:
IMG_SIZE   = (150, 150)
BATCH_SIZE = 32

# Training: normalizzazione + augmentation obbligatoria
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    shear_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Test: solo normalizzazione
test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_gen = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

CLASS_NAMES = list(train_gen.class_indices.keys())
print('Classi:', CLASS_NAMES)

In [ ]:
# Visualizza campioni con augmentation
sample_batch, _ = next(train_gen)
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(sample_batch[i])
    ax.axis('off')
plt.suptitle('Campioni con Data Augmentation', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Architettura CNN (Fase B)

In [ ]:
model = Sequential([
    # Blocco 1
    Conv2D(32,  (3,3), activation='relu', input_shape=(150, 150, 3)),
    MaxPooling2D(2, 2),

    # Blocco 2
    Conv2D(64,  (3,3), activation='relu'),
    MaxPooling2D(2, 2),

    # Blocco 3
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2, 2),

    # Blocco 4
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2, 2),

    # Livelli Dense
    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),

    # Output: 3 classi
    Dense(3, activation='softmax')
])

model.summary()

## 4. Compilazione e Training (Fase C)

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Callback personalizzata: stop se accuracy >= 98%
class StopAt98(Callback):
    def on_epoch_end(self, epoch, logs=None):
        if logs.get('accuracy', 0) >= 0.98:
            print(f'\nAccuratezza {logs["accuracy"]*100:.1f}% → training interrotto.')
            self.model.stop_training = True

history = model.fit(
    train_gen,
    epochs=20,
    validation_data=test_gen,
    callbacks=[StopAt98()]
)

## 5. Curve di Apprendimento

In [ ]:
acc     = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss    = history.history['loss']
val_loss= history.history['val_loss']
epochs  = range(1, len(acc) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, acc,     'b-o', label='Train')
ax1.plot(epochs, val_acc, 'r-o', label='Validation')
ax1.set_title('Accuracy')
ax1.set_xlabel('Epoca')
ax1.legend()

ax2.plot(epochs, loss,     'b-o', label='Train')
ax2.plot(epochs, val_loss, 'r-o', label='Validation')
ax2.set_title('Loss')
ax2.set_xlabel('Epoca')
ax2.legend()

plt.tight_layout()
plt.show()

## 6. Valutazione sul Test Set (Fase D)

In [ ]:
loss, acc = model.evaluate(test_gen)
print(f'\nTest Loss:     {loss:.4f}')
print(f'Test Accuracy: {acc*100:.2f}%')

## 7. Test con Webcam (Colab) — Domain Gap

In [ ]:
# Scatta una foto con la webcam (solo Google Colab)
try:
    from google.colab.patches import cv2_imshow
    from google.colab import output
    from IPython.display import display, Javascript
    from base64 import b64decode
    from PIL import Image
    import io

    js = Javascript('''
    async function takePhoto() {
      const div = document.createElement('div');
      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      await new Promise((resolve) => setTimeout(resolve, 2000));
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', 0.8);
    }
    ''')
    display(js)

    def take_photo():
        from IPython.display import Javascript
        from google.colab.output import eval_js
        data = eval_js('takePhoto()')
        binary = b64decode(data.split(',')[1])
        img = Image.open(io.BytesIO(binary)).convert('RGB').resize((150, 150))
        return img

    print('📷 Scatta ora (tieni la mano su sfondo chiaro)...')
    img_pil = take_photo()

    # Pre-processing identico al training
    arr = img_to_array(img_pil) / 255.0
    arr = np.expand_dims(arr, axis=0)

    pred = model.predict(arr)[0]
    idx  = np.argmax(pred)

    plt.imshow(img_pil)
    plt.axis('off')
    plt.title(f'Predizione: {CLASS_NAMES[idx].upper()}  ({pred[idx]*100:.1f}%)', fontsize=14)
    plt.show()

    print('\nConfidenza per classe:')
    for name, p in zip(CLASS_NAMES, pred):
        bar = '█' * int(p * 30)
        print(f'  {name:<10} {p*100:5.1f}%  {bar}')

except ImportError:
    print('⚠️  Webcam disponibile solo su Google Colab.')
    print('   Per testare localmente, usa il blocco qui sotto.')

## 8. Test su Immagine Locale

In [ ]:
def predict_image(path: str):
    """Carica un'immagine locale e restituisce la predizione."""
    img = load_img(path, target_size=(150, 150))
    arr = img_to_array(img) / 255.0
    arr = np.expand_dims(arr, axis=0)

    pred = model.predict(arr)[0]
    idx  = np.argmax(pred)

    plt.imshow(img)
    plt.axis('off')
    plt.title(f'{CLASS_NAMES[idx].upper()}  ({pred[idx]*100:.1f}%)', fontsize=14)
    plt.show()
    return CLASS_NAMES[idx], pred

# ── Esempio d'uso ──────────────────────────────────────────────────────────────
# predict_image('mia_mano.jpg')
print('Chiama predict_image("percorso/immagine.jpg") per testare una foto.')

In [ ]:
# Salva il modello addestrato
model.save('rps_model.keras')
print('Modello salvato → rps_model.keras')